In [1]:
import torch
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

In [2]:
import sys
sys.path.append("..")
from src.infra.gld.prof_oak_pc import LocalProfOakPcRepository

box=LocalProfOakPcRepository().load()
box

BoxEntity(name='prof_oak_pc', dataset=DatasetDict({
    train: Dataset({
        features: ['name', 'chunk', 'labels', 'input_ids', 'input_text', 'original_text', 'attention_mask'],
        num_rows: 1000
    })
}), tokenizer=PreTrainedTokenizerFast(name_or_path='/workspace/GPokeT2/data/gld/prof_oak_pc/box-latest', vocab_size=128, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[EOS]', 'pad_token': '[EOS]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
))

In [3]:
tokenizer = box.tokenizer
tokenizer

PreTrainedTokenizerFast(name_or_path='/workspace/GPokeT2/data/gld/prof_oak_pc/box-latest', vocab_size=128, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[EOS]', 'pad_token': '[EOS]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [4]:
dataset = box.dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['name', 'chunk', 'labels', 'input_ids', 'input_text', 'original_text', 'attention_mask'],
        num_rows: 1000
    })
})

In [5]:
# 6) Modelo GPT-2 enano (CPU-friendly). Ajusta si quieres aún más pequeño.
config = GPT2Config(
    vocab_size=len(tokenizer.get_vocab()),
    n_positions=4096,
    n_ctx=4096,
    n_embd=128,
    n_layer=4,
    n_head=4,
    bos_token_id=tokenizer.eos_token_id,  # type_ ignore
    eos_token_id=tokenizer.eos_token_id,  # type_ ignore
)
model = GPT2LMHeadModel(config)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(128, 128)
    (wpe): Embedding(4096, 128)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-3): 4 x GPT2Block(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=384, nx=128)
          (c_proj): Conv1D(nf=128, nx=128)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=512, nx=128)
          (c_proj): Conv1D(nf=128, nx=512)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=128, out_features=128, bias=False)
)

In [6]:
import tempfile
from typing import Any, Optional, Union
from functools import partial

import torch
import transformers
from transformers import Trainer  # type: ignore
from transformers import TrainingArguments  # type: ignore
from transformers import DataCollatorForLanguageModeling  # type: ignore


def ForCausalLMLossWeighed( # based on ForCausalLMLoss from transformers.loss.loss_utils.py
    logits: transformers.modeling_outputs.CausalLMOutputWithCrossAttentions,
    labels: torch.Tensor,
    vocab_size: int,
    num_items_in_batch: Optional[torch.Tensor] = None,
    weight_token_id=None,
    token_weight=0.05,
    ignore_index: int = -100,
    shift_labels: Optional[torch.Tensor] = None,
    **kwargs,
) -> torch.Tensor:

    logits = logits.logits.float().view(-1, vocab_size)
    labels = torch.nn.functional.pad(labels, (0, 1), value=ignore_index)
    shift_labels = labels[..., 1:].contiguous().view(-1).to(logits.device)

    weights = torch.ones((vocab_size,),device=logits.device)
    if weight_token_id is not None:
        weights[weight_token_id] = token_weight

    reduction = "sum" if num_items_in_batch is not None else "mean"
    loss = torch.nn.functional.cross_entropy(
        input=logits,
        target=shift_labels,
        weight=weights,
        ignore_index=ignore_index,
        reduction=reduction,
    )
    
    if reduction == "sum":
        # just in case users pass an int for num_items_in_batch, which could be the case for custom trainer
        if torch.is_tensor(num_items_in_batch):
            num_items_in_batch = num_items_in_batch.to(loss.device)
        loss = loss / num_items_in_batch
    
    return loss


In [ ]:
# 7) Entrenamiento ------------------------------------------------------------
# Collator causal LM (sin enmascarado aleatorio)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Consejos CPU: fp32, batch pequeño, pocos workers
training_args = TrainingArguments(
    num_train_epochs=300,  # subir si quieres más overfit
    per_device_train_batch_size=32,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=10,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_loss_func=partial(
        ForCausalLMLossWeighed,
            vocab_size=len(tokenizer.get_vocab()),
            weight_token_id=tokenizer.convert_tokens_to_ids("~"),
            token_weight=0.01,
    )
)

# Entrenar
trainer.train()

Step,Training Loss
10,0.241600


In [11]:
# 8) Generación de prueba (debería “copiar” la imagen)
with torch.no_grad():
    input_text = tokenizer("00", return_tensors="pt").to("cuda")
    out = model.generate(
        input_ids=input_text["input_ids"],
        min_length=4096,
        max_length=4096,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0].tolist(), skip_special_tokens=False)

# Restaurar saltos de línea
restored = " ".join(["\n"+c if i%64==0 else c for i,c in enumerate(decoded.split())])
print("\n=== RECONSTRUCCIÓN ===\n")
print(restored)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



=== RECONSTRUCCIÓN ===


00 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
01 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
02 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
03 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
04 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
05 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
06 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ 
07 ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~